# 03 — Train on the past and test on later races

**Under active development.** First run **python -m wearechecking history --start-year 2022 --end-year 2025**.

This notebook trains models, chooses one using 2024 only, refits through 2024, and evaluates on 2025. Test-season rolling form can use previous completed races; the model itself remains frozen.

In [ ]:
import pandas as pd
from wearechecking.history import latest_history, load_history
from wearechecking.modeling import benchmark, FEATURES

history_folder = latest_history()
results, qualifying, provenance = load_history(history_folder)
print('Source observed at:', provenance['observed_at'])
print('Feature columns:', FEATURES)
results.groupby('season').agg(events=('event_id', 'nunique'), driver_rows=('driver_id', 'size'))

## The information boundary

Features use qualifying rank plus prior driver, team and circuit performance. Current-race positions and statuses are joined as labels only after feature construction. New drivers fall back to team form. The timing tests mutate future results and verify that earlier features do not change.

In [ ]:
model_folder, report = benchmark(history_folder)
print('Selected on validation:', report['selected_model'])
display(pd.DataFrame(report['validation']).T)
print('Saved model and audit files:', model_folder.name)

## Compare against the simple baseline

Lower position MAE is better. Higher Spearman and top-three overlap are better. Model selection never uses this test table.

In [ ]:
display(pd.DataFrame({'trained_model': report['test_model'], 'qualifying_baseline': report['test_qualifying_baseline']}).T)
print('Paired MAE difference, model minus baseline:', report['paired_mae_difference_model_minus_baseline'])
print('Descriptive 95% event bootstrap interval:', report['paired_bootstrap_95_interval'])
display(pd.DataFrame(report['excluded_events']))

## Decision

The initial trained model did not beat qualifying order on held-out position MAE. Keep the simple baseline as the default. The exclusions and retrospective source corrections limit the benchmark; they are reported rather than repaired using future information.

Next research steps: penalty-aware grids, more historical seasons, weather forecasts, session-specific pace features, and an untouched later evaluation window. Do not repeatedly tune against the 2025 test set.